In [1]:
import sys
import os

# Add project root to path
sys.path.append(os.path.abspath(".."))

import pandas as pd

from src.cleaning import clean_transactions
from src.fuzzy_grouping import group_merchants

from src.eda import (
    spending_by_category,
    average_monthly_spending_by_category,
    spending_by_merchant,
    average_monthly_spending_by_merchant,
    monthly_spending,
    average_monthly_spending
)

from src.visualization import (
    plot_category_spending,
    plot_avg_monthly_category_spending,
    plot_top_merchants,
    plot_avg_monthly_merchants,
    plot_monthly_spending
)

In [2]:
df = clean_transactions("../data/Discover_Transaction_History.csv")

print(f"Rows loaded: {len(df):,}")
print(f"Unique descriptions: {df['description'].nunique():,}")

Rows loaded: 3,286
Unique descriptions: 1,403


/Users/mafphd/personalprojects/src/cleaning.py:51: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["trans_date"] = pd.to_datetime(df["trans_date"], errors="coerce")
/Users/mafphd/personalprojects/src/cleaning.py:54: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["post_date"] = pd.to_datetime(df["post_date"], errors="coerce")


In [3]:
df.head()

,trans_date,post_date,description,amount,category,desc_clean,month,day_of_week,is_weekend
0,2023-10-17,2023-10-19,WALGREENS #10196 WAUWATOSA WI,13.70,Merchandise,WALGREENS WAUWATOSA,2023-10,Tuesday,False
1,2023-10-18,2023-10-19,5-365 FOOD SERVICE TROY MI,3.05,Restaurants,FOOD SERVICE TROY MI,2023-10,Wednesday,False
2,2023-10-18,2023-10-19,QUALITY HEATING AND SHEE BROOKFIELD WI,79.65,Services,QUALITY HEATING AND SHEE BROOKFIELD,2023-10,Wednesday,False
3,2023-10-18,2023-10-19,SOMMER'S INC. MEQUON WI649149,114.59,Automotive,SOMMERS INC MEQUON,2023-10,Wednesday,False
4,2023-10-19,2023-10-19,MILTOWN EATS LLC 4144349799 WI,30.00,Supermarkets,MILTOWN EATS,2023-10,Thursday,False


In [4]:
df = group_merchants(
    df,
    threshold=85,
    max_words=2
)

print(f"Unique merchant groups: {df['merchant_clean'].nunique():,}")

Unique merchant groups: 706


In [5]:
(
    df[[
        "description",
        "desc_clean",
        "merchant_clean"
    ]]
    .sort_values("merchant_clean")
    .head(50)
)

,description,desc_clean,merchant_clean
2483,AAA ACG NE0069 EFT RCC 800-222-1134 MI,AAA ACG NE EFT RCC MI,AAA ACG
1238,AAA ACG NE0069 EFT RCC 800-222-1134 MI,AAA ACG NE EFT RCC MI,AAA ACG
45,AAA ACG NE0069 EFT RCC 800-222-1134 MI,AAA ACG NE EFT RCC MI,AAA ACG
17,AAA INSURANCE GW EFT 800-222-6424 MI,AAA INSURANCE GW EFT MI,AAA INSURANCE
1880,AAA INSURANCE GW EFT 800-222-6424 MI,AAA INSURANCE GW EFT MI,AAA INSURANCE
2466,AAA INSURANCE GW EFT 800-222-6424 MI,AAA INSURANCE GW EFT MI,AAA INSURANCE
1220,AAA INSURANCE GW EFT 800-222-6424 MI,AAA INSURANCE GW EFT MI,AAA INSURANCE
596,AAA INSURANCE GW EFT 800-222-6424 MI,AAA INSURANCE GW EFT MI,AAA INSURANCE
597,AAA INSURANCE GW EFT 800-222-6424 MI,AAA INSURANCE GW EFT MI,AAA INSURANCE
1879,AAA INSURANCE GW EFT 800-222-6424 MI,AAA INSURANCE GW EFT MI,AAA INSURANCE


In [6]:
category_totals = spending_by_category(df)

category_totals.head(20)

,category,total_spend
9,Services,42305.58
7,Merchandise,40197.25
8,Restaurants,29706.20
11,Travel/ Entertainment,26477.51
10,Supermarkets,21910.82
2,Education,7477.32
5,Home Improvement,5144.33
0,Automotive,5049.15
6,Medical Services,3224.61
4,Government Services,1857.74


In [7]:
merchant_totals = spending_by_merchant(df)

merchant_totals.head(25)

,merchant_clean,total_spend
30,APPLIANCE GALLERY,10123.51
83,CENTRAL BARKBROOKFIELD,8127.07
281,LEGACY GYM,7065.40
513,SENDIKS WAUWATOSA,6230.94
1,AAA INSURANCE,5095.03
350,MILTOWN EATS,4436.98
533,SOMMERS AUTOMOTIVE,3138.67
260,JULIES PARK,3096.68
352,MILWAUKEE ELECTRIC,2819.33
571,SPECTRUM MORC,2713.77


In [8]:
merchant_avg_monthly = average_monthly_spending_by_merchant(df)

merchant_avg_monthly.head(25)

,merchant_clean,avg_monthly_spend
30,APPLIANCE GALLERY,5061.755000
533,SOMMERS AUTOMOTIVE,3138.670000
538,SP BIRCH,1487.900000
462,PPYJULIES PARK,1203.130000
468,PY SIENA,1070.000000
260,JULIES PARK,1032.226667
1,AAA INSURANCE,1019.006000
20,AMTRAK COM,943.500000
145,DINING FURNITURE,765.730000
3,ACTCLUB SCIKIDZ,739.750000


In [ ]:
monthly_totals = monthly_spending(df)

monthly_totals

In [ ]:
overall_monthly_average = average_monthly_spending(df)

print(
    f"Average Monthly Spending: "
    f"${overall_monthly_average:,.2f}"
)

In [ ]:
plot_category_spending(
    category_totals
)

In [ ]:
plot_top_merchants(
    merchant_totals,
    top_n=20
)

In [ ]:
plot_monthly_spending(
    monthly_totals
)

In [ ]:
merchant_summary = (
    df.groupby("merchant_clean")
      .agg(
          transaction_count=("amount", "count"),
          active_months=("month", "nunique"),
          total_spend=("amount", "sum"),
          average_transaction=("amount", "mean")
      )
)

total_months = df["month"].nunique()

merchant_summary["avg_monthly_spend"] = (
    merchant_summary["total_spend"] / total_months
)

merchant_summary["monthly_frequency"] = (
    merchant_summary["active_months"] / total_months
)

merchant_summary.sort_values(
    "avg_monthly_spend",
    ascending=False
)

In [ ]:
with pd.ExcelWriter(
    "../outputs/spending_summary.xlsx",
    engine="openpyxl"
) as writer:

    category_totals.to_excel(
        writer,
        sheet_name="Category Totals",
        index=False
    )

    category_avg_monthly.to_excel(
        writer,
        sheet_name="Category Monthly Avg",
        index=False
    )

    merchant_totals.to_excel(
        writer,
        sheet_name="Merchant Totals",
        index=False
    )

    merchant_avg_monthly.to_excel(
        writer,
        sheet_name="Merchant Monthly Avg",
        index=False
    )

    monthly_totals.to_excel(
        writer,
        sheet_name="Monthly Spending",
        index=False
    )

print("Results exported.")